## Pipeline, Workflow, and Agent

A pipeline is a fixed, linear sequence of data transformations with no conditional logic;
a workflow orchestrates multiple steps with branching, retries, and dependencies;
an agent dynamically decides at runtime which tools or actions to invoke
based on the current state of the task, without a predefined execution path.

## How it actually works

These three terms are often used interchangeably, including in vendor documentation.
They describe fundamentally different execution models.

**Pipeline — pure data transformation:**
A directed acyclic graph (DAG) of transformations where data flows from input to output.
Each step is deterministic: given the same input, it always produces the same output.
There is no branching, no retries, no decisions.
Examples: scikit-learn Pipeline, Spark pipeline, a batch ETL job.
When the pipeline is done, it is done. Nothing decides the next step.

**Workflow — orchestrated execution with control flow:**
A workflow wraps pipelines and other tasks inside an orchestrator that handles:
- Sequential dependencies: step B cannot start until step A completes
- Conditional branching: if step A output meets condition X, run step B; otherwise run step C
- Parallel execution: steps with no dependencies run concurrently
- Retries and error handling: failed steps are retried with configurable backoff
- Scheduling: run at a cron schedule or on an event trigger
Examples: Apache Airflow DAGs, Prefect Flows, Azure ML Pipelines, GitHub Actions workflows.
The execution path is still defined before the workflow runs — it is just more complex than a pipeline.
The orchestrator executes a predetermined graph. It does not make decisions.

**Agent — autonomous decision-making at runtime:**
An agent has a goal (expressed in natural language or as a task specification)
and a set of tools it can invoke (APIs, code execution, search, database queries).
At each step, the agent calls a model (typically an LLM) to decide which tool to invoke next,
based on the results of prior tool calls and the original goal.
The execution path is not defined in advance. It emerges from the agent's decisions.
Examples: LangGraph agents, AutoGen agents, Claude tools-use in an agentic loop.

**The core distinction:**
A pipeline: fixed sequence, no decisions.
A workflow: fixed graph, decisions are predefined rules.
An agent: no fixed path, decisions made at runtime by the model itself.

**Where they compose:**
In mature systems, these are nested.
An agent calls a workflow as a tool.
A workflow executes multiple pipelines in sequence.
A pipeline is the leaf-level transformation.
Each layer has a different risk and observability profile.

**The ReAct pattern (Reason + Act) — how agents actually work:**
At each step the agent:
1. Reasons about the current state (Thought)
2. Selects and calls a tool (Action)
3. Observes the result (Observation)
4. Repeats until the goal is achieved or a stop condition is met
This loop is entirely controlled by the model's outputs, not by a programmer's conditional logic.

## Where it breaks or gets dangerous

**Pipelines fail predictably:**
Wrong output = wrong input or wrong transformation.
Debug by tracing the input at each step.
Failures are deterministic and reproducible.

**Workflows fail at dependency edges:**
Step B starts before step A finishes.
A retry runs a side-effectful step twice.
Conditional branching uses stale data.
Failures are structural — fixable by reviewing the DAG definition.

**Agents fail non-deterministically — and this is the fundamental risk:**

1. **Unbounded execution:** the agent can loop indefinitely.
   Every agent loop needs a maximum step count or a timeout.
   Without it, a stuck agent runs until you hit a rate limit or cost limit.

2. **Tool misuse:** the agent calls a write tool when it should have called a read tool.
   It deletes a record trying to update it. It sends an email when it was asked to draft one.
   Tool design is a safety concern. Destructive tools must require explicit confirmation.

3. **Prompt injection:** external content observed by the agent contains instructions.
   A document the agent reads tells it to exfiltrate data.
   The agent, following the instruction, does it.
   Agents must treat all observed content as data, never as instructions.

4. **Compounding errors:** in a 10-step agent loop, a wrong decision at step 3
   propagates through every subsequent step.
   The final output is confidently wrong.
   Unlike a pipeline, there is no natural checkpoint where the error surfaces.

5. **Non-reproducibility:** the same prompt + same tools + same initial state
   may produce a different execution path due to model temperature.
   This is fundamentally incompatible with audit requirements in regulated contexts.
   For enterprise use cases requiring reproducibility: use deterministic workflows, not agents.

In [ ]:
# Pipeline vs Workflow vs Agent — structural comparison

import time

# ── 1. PIPELINE ──────────────────────────────────────────────────────────────
# Fixed transformation sequence. No decisions. Same input always yields same output.

def extract(raw_data):
    return [row.strip() for row in raw_data if row.strip()]

def transform(rows):
    return [row.upper() for row in rows]

def load(rows):
    return {"status": "loaded", "rows": len(rows), "sample": rows[:2]}

def run_pipeline(raw):
    return load(transform(extract(raw)))

print("=" * 60)
print("PIPELINE: fixed transformation chain")
raw = ["  invoice 1001  ", "", "  invoice 1002  ", "  invoice 1003  "]
result = run_pipeline(raw)
print(f"  Input rows: {len(raw)}, Output rows: {result['rows']}")
print(f"  Sample: {result['sample']}")
print("  Deterministic. No branching. No decisions.")


# ── 2. WORKFLOW ───────────────────────────────────────────────────────────────
# Steps with dependencies, conditional branching, and retry logic.

class WorkflowStep:
    def __init__(self, name, fn, max_retries=2):
        self.name = name
        self.fn = fn
        self.max_retries = max_retries

    def run(self, *args, **kwargs):
        for attempt in range(1, self.max_retries + 2):
            try:
                result = self.fn(*args, **kwargs)
                print(f"  Step '{self.name}': SUCCESS (attempt {attempt})")
                return result
            except Exception as e:
                print(f"  Step '{self.name}': FAILED attempt {attempt} — {e}")
                if attempt > self.max_retries:
                    raise
                time.sleep(0.01)

attempt_counter = {"validate": 0}

def validate_data(data):
    attempt_counter["validate"] += 1
    if attempt_counter["validate"] < 3:
        raise ValueError("Validation service temporarily unavailable")
    return {"valid": True, "rows": len(data)}

def route_based_on_size(validation_result):
    if validation_result["rows"] > 100:
        return "batch_process"
    return "realtime_process"

print("\n" + "=" * 60)
print("WORKFLOW: orchestrated steps with retry and conditional branching")
validate_step = WorkflowStep("validate_data", validate_data, max_retries=3)
validation_result = validate_step.run(["row"] * 50)
route = route_based_on_size(validation_result)
print(f"  Conditional branch taken: '{route}' (rows={validation_result['rows']})")
print("  Execution path: predefined in the DAG. Decisions are rules, not model outputs.")


# ── 3. AGENT ─────────────────────────────────────────────────────────────────
# ReAct loop: model decides which tool to call at each step.

class SimpleAgent:
    """
    Simulates the ReAct pattern without an actual LLM.
    In production: each 'decide' call is an LLM inference.
    The model receives the task, tool results so far, and returns the next action.
    """

    def __init__(self, tools, max_steps=8):
        self.tools = tools
        self.max_steps = max_steps
        self.steps_taken = 0

    def decide(self, task, observations):
        """Simulate model decision. In real agents: LLM call here."""
        if not observations:
            return "search_vendor", {"vendor_id": "V-042"}
        if "vendor_name" in observations and "credit_limit" not in observations:
            return "fetch_credit_limit", {"vendor_id": "V-042"}
        if "credit_limit" in observations and "open_invoices" not in observations:
            return "list_open_invoices", {"vendor_id": "V-042"}
        return None, None  # task complete

    def run(self, task):
        print(f"  Task: {task}")
        observations = {}
        for step in range(1, self.max_steps + 1):
            action, params = self.decide(task, observations)
            if action is None:
                print(f"  Step {step}: DONE — goal achieved")
                break
            result = self.tools[action](**params)
            observations.update(result)
            print(f"  Step {step}: called '{action}({params})' → {result}")
            self.steps_taken += 1
        else:
            print(f"  STOPPED: max_steps={self.max_steps} reached — safety guardrail fired")
        return observations


tools = {
    "search_vendor": lambda vendor_id: {"vendor_name": "Contoso Ltd", "vendor_id": vendor_id},
    "fetch_credit_limit": lambda vendor_id: {"credit_limit": 250000, "currency": "USD"},
    "list_open_invoices": lambda vendor_id: {"open_invoices": 3, "total_open": 47800},
}

print("\n" + "=" * 60)
print("AGENT: ReAct loop — tool selection decided at runtime")
agent = SimpleAgent(tools=tools, max_steps=8)
final_state = agent.run(task="Summarise vendor V-042 credit exposure")
print(f"  Final state: {final_state}")
print("  Execution path: not predefined. Model chose each tool call.")
print("  Non-deterministic. Different model temperature = different path.")

## My D365 analogy

The pipeline-workflow-agent distinction maps cleanly to three modes of operation
that any D365 functional consultant recognises.

A pipeline is the D365 batch job.
The MRP explosion runs the same calculation every time on the same inputs.
No decisions. No branching. Pure transformation.
Demand forecast in, planned orders out.

A workflow is the D365 workflow engine — purchase order approval, invoice matching,
journal approval.
It has defined routing rules: amount above threshold goes to finance director,
below threshold goes to department head.
The branching logic is configured in the workflow designer.
The workflow does not decide — it follows configured rules.
Three-way matching is a workflow: receipt posted, invoice received, PO confirmed —
only if all three match does the invoice proceed to payment.

An agent is what you hire a senior AP consultant to do when the rules break down.
An invoice arrives that does not match the PO.
The three-way match fails.
A human investigates: fetches the original PO, calls the vendor,
checks the goods receipt, decides whether to create a credit note or request a revised invoice.
Each step is decided based on the result of the previous one.
There is no predefined path.

Building an AI agent to handle D365 exception cases is exactly this:
replace the senior AP consultant's decision loop with an LLM-driven tool loop.
With the same caveats: you would not give the consultant
write access to production without an approval gate,
and you would not give the agent one either.

## LinkedIn post idea

After 14 years in D365, I have seen three modes of automation:

Batch jobs that run the same calculation every night.
Workflow engines that route transactions based on configured rules.
Consultants called in when neither of those handles the exception.

In AI engineering, these are called pipeline, workflow, and agent.
Different terms. Same conceptual boundary.

The thing that took me a while to understand:
an agent is not just a smarter workflow.
A workflow follows a predefined graph — conditions, branches, retries.
An agent decides at runtime which tool to call next,
based on what it just observed.

That is the key difference. And it is also where the risk lives.

A misconfigured workflow takes the wrong branch.
You fix the configuration.
A misconfigured agent takes a different wrong path every time it runs.
You need guardrails, not just configuration.

The more autonomous the system, the more deliberately you have to constrain it.
Senior D365 consultants know this.
We just did not have a word for it until now.